# 02 · MLflow on top of OpenClaw

Notebook 01 drove OpenClaw from Python. Here we put MLflow around it so that every answer becomes a **trace**:
* what the agent was asked;
* which instructions it ran with;
* every model turn, with tokens and cost;
* every file it opened;
* what it answered.

Traces are what the rest of the loop (signal → store → update → gate) is built on.

```
acme_support_agent (AGENT)            tags: topic, runtime, prompt_version
  └─ openclaw.run (CHAIN)             run id, session key
       ├─ claude-haiku-5-5 (CHAT_MODEL)   tokens, cost, tool calls requested    <- chat.history
       ├─ read (TOOL)                     {"path": "knowledge/account_access.md"} <- live tool events
       └─ claude-haiku-5-5 (CHAT_MODEL)
```

In [1]:
from sia import notebook as nb

nb.setup(db="sdk.db", experiment="acme-support-sdk")

2026/10/20 07:28:29 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/20 07:28:29 INFO mlflow.store.db.utils: Updating database tables


2026/10/20 07:28:29 INFO mlflow.tracking.fluent: Experiment with name 'acme-support-sdk' does not exist. Creating a new experiment.


tracking_uri  sqlite:////Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/sdk.db
experiment    acme-support-sdk (id 1)
backend       offline
agent_model   anthropic:/claude-haiku-5-5
judge_model   anthropic:/claude-opus-5-5
llm_judges    True


{'tracking_uri': 'sqlite:////Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/sdk.db',
 'experiment': 'acme-support-sdk (id 1)',
 'backend': 'offline',
 'agent_model': 'anthropic:/claude-haiku-5-5',
 'judge_model': 'anthropic:/claude-opus-5-5',
 'llm_judges': True}

In [2]:
from sia.openclaw_client import Gateway

gw = Gateway(nb.NB_STORE / "sdk" / "gateway", nb.sdk_workspace("openclaw")).start()   # reuses a running one
print(gw.url, "· pid", gw.pid(), "· workspace", gw.workspace)

ws://127.0.0.1:19556 · pid 30863 · workspace /Users/jon/Documents/Claude/ODSC West 2026/self-improving-agent/notebooks/.mlflow/sdk/openclaw


## 1. The instructions come from the Prompt Registry

`AGENTS.md` is versioned in MLflow's Prompt Registry, and production is whatever the `@production` alias points
at. The agent's workspace is just where the current version gets deployed: `set_agents_md()` pushes a prompt
version into the running agent with the SDK's `set_file`.

In [3]:
import mlflow
from sia import config
from sia.openclaw_client import OpenClawAgent

nb.ensure_production()                        # registers the flawed v1 and points @production at it, once
prompt = mlflow.genai.load_prompt(f"prompts:/{config.PROMPT_NAME}@production")

openclaw = OpenClawAgent(gw)                  # an openclaw-sdk client on a background event loop
openclaw.set_agents_md(prompt.template)
assert openclaw.get_agents_md() == prompt.template
print(f"OpenClaw is running {config.PROMPT_NAME} v{prompt.version}")

OpenClaw is running acme-support-agents-md v1


## 2. Step by step: a trace around one turn

The simplest useful thing: open a span, run the turn inside it. MLflow records the question, the answer, and how
long it took. Below, `openclaw.run()` executes an SDK coroutine on the client's event loop and waits for it.

In [4]:
from mlflow.entities import SpanType

QUESTION = "Can we set up SSO with Okta on our Pro plan?"

with mlflow.start_span("acme_support_agent", span_type=SpanType.AGENT) as root:
    root.set_inputs({"question": QUESTION})
    result = openclaw.run(openclaw.agent().execute(QUESTION))
    root.set_outputs(result.content)

nb.show(nb.last_trace())

AGENT      acme_support_agent      5.5s  'Policies can vary by plan and region, so please double-check the Terms'


That trace tells us *what* the agent said, but not *how*. Did it open the policy file? How many model calls
did it make? What did they cost?

### Tool calls: a callback handler

`MLflowTracer` is an openclaw-sdk `CallbackHandler`. Each `tool` event with `phase: start` opens a TOOL span, and
the matching `phase: result` closes it. Both use the gateway's timestamps. The handler builds spans with
`mlflow.start_span_no_context(parent_span=...)`, because the SDK's callbacks run on its event loop's thread, not
the thread that opened the trace.

In [5]:
from sia import openclaw_client

import inspect

print(inspect.getsource(openclaw_client.MLflowTracer))

class MLflowTracer(_callback_base()):
    """openclaw-sdk callbacks -> MLflow TOOL spans, live, under `parent`.

    The gateway streams `agent` events; those with stream == "tool" carry
    {phase: "start" | "result", toolCallId, name, args | result, isError} plus a timestamp.
    """

    def __init__(self, parent):
        self.parent = parent
        self.open: dict[str, Any] = {}
        self.events: list[dict] = []  # every raw event, for inspection in the notebooks

    async def on_stream_event(self, agent_id: str, event) -> None:
        payload = event.data.get("payload") or {}
        self.events.append({"event": event.data.get("event"), **payload})
        if payload.get("stream") != "tool":
            return
        data = payload.get("data") or {}
        call_id, ts = data.get("toolCallId"), _ns(payload.get("ts"))
        if data.get("phase") == "start":
            self.open[call_id] = mlflow.start_span_no_context(
                data.get("name") or "tool", span_type=

### Model calls: the session transcript

The event stream has no per-call token counts. The session's `chat.history` does. After the run,
`add_llm_spans()` adds one CHAT_MODEL span per assistant turn of *this* run. Each span carries:
* the timestamps OpenClaw recorded;
* the model and provider;
* MLflow's standard token-usage attribute (`mlflow.chat.tokenUsage`);
* cost (`mlflow.llm.cost`).

MLflow then rolls usage and cost up to the trace.

In [6]:
print(nb.source(openclaw_client, "add_llm_spans"))

def add_llm_spans(parent, history: list[dict], run_id: str) -> list[dict]:
    """One CHAT_MODEL span per assistant turn of this run, from the session's chat.history."""
    turns = []
    for i, msg in enumerate(history):
        meta = msg.get("__openclaw") or {}
        if msg.get("role") != "assistant" or meta.get("runId") != run_id:
            continue
        usage = msg.get("usage") or {}
        cost = usage.get("cost") or {}
        blocks = msg.get("content") or []
        calls = [{"name": b.get("name"), "arguments": b.get("arguments")} for b in blocks
                 if isinstance(b, dict) and b.get("type") == "toolCall"]
        cached = (usage.get("cacheRead") or 0) + (usage.get("cacheWrite") or 0)
        attrs = {
            SpanAttributeKey.MODEL: msg.get("model"), SpanAttributeKey.MODEL_PROVIDER: msg.get("provider"),
            SpanAttributeKey.CHAT_USAGE: {
                TokenUsageKey.INPUT_TOKENS: (usage.get("input") or 0) + cached,
                TokenUsageK

## 3. All together: `OpenClawAgent.ask()`

`ask()` is those three steps, plus tags. It's an ordinary synchronous function, which is exactly what
`mlflow.genai.evaluate()` needs as a `predict_fn` in notebook 04.

v1's instructions discourage opening files, and on its own the agent doesn't (section 5 shows what that costs).
To see TOOL spans, this question names the policy file.

In [7]:
LOOKUP = "Please read knowledge/account_access.md and tell me: can we set up SSO with Okta on our Pro plan?"
answer = openclaw.ask(LOOKUP, tags={"topic": "account_access", "runtime": "openclaw",
                                   "prompt_version": str(prompt.version)})
trace = nb.last_trace()
nb.show(trace)

AGENT      acme_support_agent      5.4s  "No, SSO with Okta isn't available on the Pro plan. According to `knowl"
  └─ CHAIN      openclaw.run            5.4s  
    └─ CHAT_MODEL claude-haiku-5-5        0.8s  -> tools [{'name': 'read', 'arguments': {'path': 'knowledge/account_access.md', 'optional': True}}]
    └─ TOOL       read                    0.0s  knowledge/account_access.md
    └─ CHAT_MODEL claude-haiku-5-5        3.1s  -> "No, SSO with Okta isn't available on the Pro plan. According to `knowl"
tags: {'topic': 'account_access', 'runtime': 'openclaw', 'prompt_version': '1'}


In [8]:
import pandas as pd

print("trace token usage:", trace.info.token_usage)
pd.DataFrame([{
    "span": s.name, "type": s.span_type, "seconds": round((s.end_time_ns - s.start_time_ns) / 1e9, 2),
    "input tokens": (s.attributes.get("mlflow.chat.tokenUsage") or {}).get("input_tokens"),
    "output tokens": (s.attributes.get("mlflow.chat.tokenUsage") or {}).get("output_tokens"),
    "cost $": (s.attributes.get("mlflow.llm.cost") or {}).get("total_cost"),
    "detail": s.inputs.get("path") if s.span_type == "TOOL" and isinstance(s.inputs, dict) else "",
} for s in sorted(trace.data.spans, key=lambda s: s.start_time_ns)])

trace token usage: {'input_tokens': 17523, 'output_tokens': 522, 'total_tokens': 18045, 'cache_read_input_tokens': 15421, 'cache_creation_input_tokens': 2096}


,span,type,seconds,input tokens,output tokens,cost $,detail
0,acme_support_agent,AGENT,5.36,NaN,NaN,NaN,
1,openclaw.run,CHAIN,5.36,NaN,NaN,NaN,
2,claude-haiku-5-5,CHAT_MODEL,0.77,8599.0,76.0,0.000328,
3,read,TOOL,0.01,NaN,NaN,NaN,knowledge/account_access.md
4,claude-haiku-5-5,CHAT_MODEL,3.12,8924.0,446.0,0.000350,


Open the MLflow UI (`make notebooks-ui`, then the *acme-support-sdk* experiment) to see the same trace as a
timeline. The model calls show their token counts and cost, and the `read` spans show the file contents the
agent got back.

## 4. Link the trace to the prompt version

Tags make traces searchable. A **prompt link** goes one step further: the prompt version page in the Registry
lists every trace produced with it, so a regression can be traced back to the instructions that caused it.

In [9]:
mlflow.MlflowClient().link_prompt_versions_to_trace(prompt_versions=[prompt], trace_id=trace.info.trace_id)
{k: v for k, v in mlflow.get_trace(trace.info.trace_id).info.tags.items() if "prompt" in k.lower()}

{'prompt_version': '1',
 'mlflow.linkedPrompts': '[{"name": "acme-support-agents-md", "version": "1"}]'}

## 5. Signal on a trace

The trace is also where verdicts go. Three kinds:
* **Code checks** look at the answer, or at the TOOL spans: did it read `knowledge/account_access.md`?
  Here the customer asks plainly, and v1 answers without opening anything.
* **A reviewer**, here one of the scenario's simulated support leads, leaves a pass/fail with a rationale, plus
  the correct answer.
* **LLM judges** (Claude Opus 5.5) come in notebook 04.

Each is an MLflow *assessment* on the trace, with its source.

In [10]:
from mlflow.entities import AssessmentSource, AssessmentSourceType
from sia import reviewers, signals
from sia.scenario import case_for_question

answer = openclaw.ask(QUESTION, tags={"topic": "account_access", "runtime": "openclaw",
                                     "prompt_version": str(prompt.version)})    # as a customer would ask it
trace = nb.last_trace()
print(answer, "\n")

code_source = AssessmentSource(source_type=AssessmentSourceType.CODE, source_id="sia.signals")
checks = [(signals.no_stale_policy, {"outputs": answer}), (signals.no_disclaimer_opener, {"outputs": answer}),
          (signals.read_right_policy, {"trace": trace}), (signals.focused_reading, {"trace": trace})]
for check, kwargs in checks:
    fb = check(**kwargs)
    mlflow.log_feedback(trace_id=trace.info.trace_id, name=check.name, value=fb.value,
                        rationale=fb.rationale, source=code_source)

verdict = reviewers.log_review(trace.info.trace_id, case_for_question(QUESTION), answer)
t = mlflow.get_trace(trace.info.trace_id)
pd.DataFrame([{"name": a.name, "value": a.value, "source": str(a.source.source_type).split(".")[-1],
               "rationale": (a.rationale or "")[:90]} for a in t.info.assessments if a.feedback is not None])

Policies can vary by plan and region, so please double-check the Acme Cloud Terms of Service and your plan details before you rely on this.

I don't have information in my quick reference about whether SSO or Okta integration is included on the Pro plan, so I can't confirm it's available. My best guess is that SSO is usually offered on higher-tier plans, so it may not be included on Pro. Please contact Acme Cloud support or your account representative to confirm, and ask whether:

- SSO with Okta (SAML or OIDC) is available on your Pro plan, or what plan you'd need.
- There's an add-on or upgrade path, and what it costs.
- Any setup steps or prerequisites apply on the Okta side.

In the meantime, here's what I can help with from my quick reference:

- **Inviting teammates:** Workspace → Members → Invite.
- **Account access:** Password reset links are valid for 24 hours.
- **Refunds:** Available within 30 days of purchase. 



,name,value,source,rationale
0,no_stale_policy,False,CODE,Quotes outdated policy: refund window is 14 da...
1,no_disclaimer_opener,False,CODE,Opens with a disclaimer: 'Policies can vary by...
2,read_right_policy,False,CODE,Answered without reading any policy file (expe...
3,focused_reading,True,CODE,Read 0 policy file(s): none.
4,quality,False,HUMAN,Missing a key fact (expected something matchin...


## 6. A little production traffic

A few more customer questions, each tagged `source=production`. Once traces are tagged and labeled, finding
"every OpenClaw answer a reviewer failed" is a search filter.

In [11]:
from sia.scenario import load_cases

for case in load_cases("traffic")[::4]:
    ans = openclaw.ask(case.question, tags={"topic": case.topic, "runtime": "openclaw", "source": "production",
                                            "prompt_version": str(prompt.version)})
    reviewers.log_review(nb.last_trace().info.trace_id, case, ans)
    print(f"[{case.topic}] {case.question}\n  -> {ans[:110]}...\n")

[billing] How long can I wait to request a refund?
  -> Policies can vary by plan and region, so please double-check the Terms of Service too.

Refunds are available ...



[billing] What happens if my card payment fails?
  -> Policies can vary by plan and region, so please double-check the Terms of Service for the exact rules that app...



[account_access] I lost my phone with my authenticator app. How do I get back in?
  -> I'm sorry, losing your phone with your authenticator app is stressful. Policies can vary by plan and region, s...



[product_howto] Does Acme Cloud integrate with Jira?
  -> Thanks for reaching out! Policies can vary by plan and region, so please double-check the Acme Cloud Terms of ...



In [12]:
import json

mlflow.flush_trace_async_logging()
failed = mlflow.search_traces(filter_string="tags.runtime = 'openclaw' AND feedback.quality = 'false'",
                              return_type="list")
pd.DataFrame([{"topic": t.info.tags.get("topic"), "question": json.loads(t.data.request)["question"][:60],
               "reviewer says": next(a.rationale for a in t.info.assessments if a.name == "quality")[:120]}
              for t in failed])

,topic,question,reviewer says
0,product_howto,Does Acme Cloud integrate with Jira?,Missing a key fact (expected something matchin...
1,account_access,I lost my phone with my authenticator app. How...,Missing a key fact (expected something matchin...
2,billing,What happens if my card payment fails?,Missing a key fact (expected something matchin...
3,billing,How long can I wait to request a refund?,Stated something wrong or outdated ('30 day')....
4,account_access,Can we set up SSO with Okta on our Pro plan?,Missing a key fact (expected something matchin...


In [13]:
openclaw.close()

**Next:** 03 · Hermes Agent from Python. A second runtime, the same trace shape.